<a href="https://colab.research.google.com/github/leonleonwang88-a11y/financial-rag-challenge/blob/main/FinancialRAG_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Install required packages
!pip install -q huggingface_hub pandas chromadb sentence-transformers openai

# Step 2: Import tools
from huggingface_hub import snapshot_download, login
from google.colab import userdata

# Step 3: Load token from Colab's secret vault (safe!)
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# Step 4: Download ONLY the .txt files
snapshot_download(
    repo_id="databricks/officeqa",
    repo_type="dataset",
    allow_patterns="treasury_bulletins_parsed/transformed/*.txt",
    local_dir="./data"
)

# Step 5: Download the answer key CSV
snapshot_download(
    repo_id="databricks/officeqa",
    repo_type="dataset",
    allow_patterns="officeqa_full.csv",
    local_dir="./data"
)

print("✅ Download complete!")

In [ ]:
import os

# Check the text files
txt_folder = "./data/treasury_bulletins_parsed/transformed"
txt_files = sorted(os.listdir(txt_folder))
print(f"📄 Total .txt files: {len(txt_files)}")
print(f"📄 First 3 files: {txt_files[:3]}")
print(f"📄 Last 3 files: {txt_files[-3:]}")

# Check the CSV
csv_path = "./data/officeqa_full.csv"
print(f"\n📊 CSV exists: {os.path.exists(csv_path)}")
if os.path.exists(csv_path):
    size_kb = os.path.getsize(csv_path) / 1024
    print(f"📊 CSV size: {size_kb:.2f} KB")

In [ ]:
import pandas as pd

# Load the answer key
df = pd.read_csv("./data/officeqa_full.csv")
print(f"📊 Total questions in CSV: {len(df)}")

recent_years = ['2022', '2023', '2024', '2025']
pattern = '|'.join(recent_years)

# Strategy: Keep any question where AT LEAST ONE source file is from 2022-2025
df_filtered = df[df['source_files'].str.contains(pattern, na=False)].copy()

# Count how many source files each question references
def count_sources(s):
    if pd.isna(s):
        return 0
    return len([f for f in str(s).split('\n') if f.strip()])

# Check if ALL sources are in recent years (strict) vs ANY (loose)
def all_in_recent(s):
    if pd.isna(s):
        return False
    files = [f.strip() for f in str(s).split('\n') if f.strip()]
    return all(any(yr in f for yr in recent_years) for f in files)

df_filtered['num_sources'] = df_filtered['source_files'].apply(count_sources)
df_filtered['all_recent'] = df_filtered['source_files'].apply(all_in_recent)

print(f"\n📊 Questions touching 2022-2025 (any source): {len(df_filtered)}")
print(f"📊 Difficulty breakdown:")
print(df_filtered['difficulty'].value_counts())
print(f"\n📊 Of these, fully answerable from 2022-2025 only: {df_filtered['all_recent'].sum()}")

# RECOMMENDED: Keep only questions where ALL sources are from 2022-2025
# This guarantees the answer CAN be found in our subset
df_final = df_filtered[df_filtered['all_recent']].copy()

print(f"\n✅ Final dataset for testing: {len(df_final)} questions")
print(f"📊 By difficulty:")
print(df_final['difficulty'].value_counts())

# Show samples
print(f"\n📋 Sample EASY questions:")
easy = df_final[df_final['difficulty'] == 'easy'].head(3)
for i, row in easy.iterrows():
    print(f"\n--- {row['uid']} ({row['difficulty']}) ---")
    print(f"Q: {row['question'][:200]}")
    print(f"A: {row['answer']}")
    print(f"Source: {row['source_files']}")

print(f"\n📋 Sample HARD questions:")
hard = df_final[df_final['difficulty'] == 'hard'].head(3)
for i, row in hard.iterrows():
    print(f"\n--- {row['uid']} ({row['difficulty']}) ---")
    print(f"Q: {row['question'][:200]}")
    print(f"A: {row['answer']}")
    print(f"Source: {row['source_files']}")

# Save
df_final.to_csv("./data/filtered_questions.csv", index=False)
print(f"\n💾 Saved {len(df_final)} questions to ./data/filtered_questions.csv")

In [ ]:
import pandas as pd

# Load the answer key
df = pd.read_csv("./data/officeqa_full.csv")
print(f"📊 Total questions in CSV: {len(df)}")

# EXPANDED: 8 recent years instead of 4
recent_years = ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
pattern = '|'.join(recent_years)

df_touching = df[df['source_files'].str.contains(pattern, na=False)].copy()

def all_in_recent(s):
    if pd.isna(s):
        return False
    files = [f.strip() for f in str(s).split('\n') if f.strip()]
    return all(any(yr in f for yr in recent_years) for f in files)

df_final = df_touching[df_touching['source_files'].apply(all_in_recent)].copy()

print(f"\n📊 Questions touching 2018-2025: {len(df_touching)}")
print(f"✅ Fully answerable from 2018-2025: {len(df_final)}")
print(f"\n📊 By difficulty:")
print(df_final['difficulty'].value_counts())

# Save
df_final.to_csv("./data/filtered_questions.csv", index=False)
print(f"\n💾 Saved {len(df_final)} questions to ./data/filtered_questions.csv")

# Show a few samples
print(f"\n📋 Sample EASY questions:")
for i, row in df_final[df_final['difficulty']=='easy'].head(3).iterrows():
    print(f"\n--- {row['uid']} ---")
    print(f"Q: {row['question'][:180]}")
    print(f"A: {row['answer']}")
    print(f"Source: {row['source_files']}")

In [ ]:
import pandas as pd
import os

# Load our filtered questions
df = pd.read_csv("./data/filtered_questions.csv")

# Extract all unique .txt filenames referenced by our questions
all_files = set()
for sources in df['source_files'].dropna():
    for f in str(sources).split('\n'):
        f = f.strip()
        if f.endswith('.txt'):
            all_files.add(f)

all_files = sorted(all_files)
print(f"📂 Unique source files needed: {len(all_files)}")
print(f"\n📄 Files:")
for f in all_files:
    print(f"  - {f}")

# Verify all these files exist on disk
txt_folder = "./data/treasury_bulletins_parsed/transformed"
existing = os.listdir(txt_folder)
missing = [f for f in all_files if f not in existing]
if missing:
    print(f"\n⚠️ Missing files: {missing}")
else:
    print(f"\n✅ All {len(all_files)} required files exist on disk!")

# Save for later use
with open("./data/needed_files.txt", "w") as f:
    f.write("\n".join(all_files))
print(f"💾 Saved file list to ./data/needed_files.txt")

In [ ]:
import os
import glob

# Find all .txt files from 2018-2025
recent_years = ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
txt_folder = "./data/treasury_bulletins_parsed/transformed"

all_files = []
for year in recent_years:
    pattern = os.path.join(txt_folder, f"treasury_bulletin_{year}_*.txt")
    all_files.extend(glob.glob(pattern))

all_files = sorted(all_files)
print(f"📂 Total .txt files to process (2018-2025): {len(all_files)}")

# BASELINE chunking: simple 500-character chunks, NO overlap
chunks = []
for filepath in all_files:
    filename = os.path.basename(filepath)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()

    # Simple split every 500 characters
    for i in range(0, len(text), 500):
        chunk_text = text[i:i+500]
        if len(chunk_text.strip()) > 50:  # Skip tiny chunks
            chunks.append({
                "text": chunk_text,
                "source": filename,
                "chunk_id": f"{filename}_{i}"
            })

print(f"📦 Total chunks created: {len(chunks)}")
print(f"\n📋 Sample chunk:")
print(f"Source: {chunks[0]['source']}")
print(f"Text (first 200 chars): {chunks[0]['text'][:200]}...")

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# Load embedding model (downloads ~80MB on first run, ~30 seconds)
print("📥 Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model ready!")

# Create a fresh ChromaDB collection
client = chromadb.Client()

# Delete old collection if it exists (so we can re-run safely)
try:
    client.delete_collection("baseline_rag")
except:
    pass

collection = client.create_collection(
    name="baseline_rag",
    metadata={"description": "Baseline: 500-char chunks, no overlap, no metadata filter"}
)

# Embed and store chunks in batches (faster + avoids memory issues)
BATCH_SIZE = 256
print(f"\n🔄 Embedding and storing {len(chunks)} chunks...")

for i in tqdm(range(0, len(chunks), BATCH_SIZE)):
    batch = chunks[i:i+BATCH_SIZE]
    texts = [c["text"] for c in batch]

    # Create embeddings
    embeddings = embed_model.encode(texts, show_progress_bar=False).tolist()

    # Add to ChromaDB
    collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=[{"source": c["source"]} for c in batch],
        ids=[c["chunk_id"] for c in batch]
    )

print(f"\n✅ Baseline vector database ready!")
print(f"📊 Total chunks in DB: {collection.count()}")

# Quick test query
test_query = "What was U.S. Treasury investment in Japanese Yen?"
print(f"\n🧪 Test query: '{test_query}'")
test_embedding = embed_model.encode([test_query]).tolist()
results = collection.query(query_embeddings=test_embedding, n_results=3)

print(f"\n🎯 Top 3 results:")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"\n--- Result {i+1} ---")
    print(f"Source: {meta['source']}")
    print(f"Text (first 200 chars): {doc[:200]}...")

In [ ]:
import pandas as pd

# Load filtered questions
df_questions = pd.read_csv("./data/filtered_questions.csv")
print(f"📋 Testing {len(df_questions)} questions on baseline RAG...\n")

# Helper: parse the ground truth source files for a question
def get_gt_files(source_str):
    """Returns a set of source filenames that are the 'ground truth' for a question."""
    if pd.isna(source_str):
        return set()
    return set(f.strip() for f in str(source_str).split('\n') if f.strip().endswith('.txt'))

# Run all queries and compute metrics
hit_count = 0
mrr_sum = 0.0
results_log = []

K = 5  # Top-K retrieval

for idx, row in df_questions.iterrows():
    question = row['question']
    gt_files = get_gt_files(row['source_files'])

    # Embed the question and search
    q_embedding = embed_model.encode([question]).tolist()
    results = collection.query(query_embeddings=q_embedding, n_results=K)

    # Get the unique source files from top-K results (in order)
    retrieved_sources = []
    seen = set()
    for meta in results['metadatas'][0]:
        src = meta['source']
        if src not in seen:
            retrieved_sources.append(src)
            seen.add(src)

    # HIT RATE@5: Did ANY ground truth file appear in top-5?
    hit = any(gt in retrieved_sources for gt in gt_files)
    if hit:
        hit_count += 1

    # MRR: Rank of FIRST correct ground truth file in retrieved list
    rank = 0
    for i, src in enumerate(retrieved_sources, start=1):
        if src in gt_files:
            rank = i
            break
    reciprocal_rank = 1.0 / rank if rank > 0 else 0.0
    mrr_sum += reciprocal_rank

    results_log.append({
        "uid": row['uid'],
        "difficulty": row['difficulty'],
        "gt_files": list(gt_files),
        "retrieved": retrieved_sources,
        "hit": hit,
        "rank_of_first_correct": rank,
        "reciprocal_rank": reciprocal_rank
    })

# Final scores
hit_rate = hit_count / len(df_questions)
mrr = mrr_sum / len(df_questions)

print(f"{'='*60}")
print(f"📊 BASELINE METRICS (Set A: Retriever)")
print(f"{'='*60}")
print(f"Hit Rate@{K}:  {hit_rate:.2%}  ({hit_count}/{len(df_questions)} questions)")
print(f"MRR:          {mrr:.4f}")
print(f"{'='*60}\n")

# Show per-question detail
print("📋 Per-question breakdown:")
for r in results_log:
    status = "✅" if r['hit'] else "❌"
    print(f"{status} {r['uid']} ({r['difficulty']}): rank={r['rank_of_first_correct']}, GT={r['gt_files']}")

# Save results
pd.DataFrame(results_log).to_csv("./data/baseline_retrieval_results.csv", index=False)
print(f"\n💾 Saved detailed results to ./data/baseline_retrieval_results.csv")

In [ ]:
# Install Gemini library
!pip install -q google-generativeai

import google.generativeai as genai
from google.colab import userdata

# Load API key from Colab Secrets
gemini_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=gemini_key)

# Test the model
model = genai.GenerativeModel('gemini-1.5-flash')
response = model.generate_content("Say 'Hello, RAG project!' if you can hear me.")
print("🤖 Gemini says:", response.text)

In [ ]:
# Install the NEW Gemini library
!pip install -q -U google-genai

from google import genai
from google.colab import userdata

# Load API key from Colab Secrets
gemini_key = userdata.get('GEMINI_API_KEY')

# Create the client
client = genai.Client(api_key=gemini_key)

# Test with the current model name
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Say 'Hello, RAG project!' if you can hear me."
)
print("🤖 Gemini says:", response.text)

In [ ]:
!pip install -q -U google-genai

from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

print("📋 Models available to your account:\n")
for m in client.models.list():
    # Only show models that can generate text
    if 'generateContent' in (m.supported_actions or []):
        print(f"  ✅ {m.name}")

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

# Replace MODEL_NAME_HERE with one from your list above
# Remove the "models/" prefix if present
response = client.models.generate_content(
    model="gemini-flash-latest",  # 👈 TRY THIS FIRST
    contents="Say 'Hello, RAG project!' if you can hear me."
)
print("🤖 Gemini says:", response.text)

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",  # 👈 Less popular = less busy
    contents="Say 'Hello, RAG project!' if you can hear me."
)
print("🤖 Gemini says:", response.text)

In [ ]:
from google import genai
from google.colab import userdata
import time

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

# List of models to try, in order of preference
MODELS_TO_TRY = [
    "gemini-2.5-flash-lite",
    "gemini-2.0-flash-lite",
    "gemini-2.0-flash-001",
    "gemini-2.5-flash",
    "gemini-flash-lite-latest",
]

def call_gemini(prompt, max_retries=3):
    """Call Gemini with automatic model fallback and retries."""
    for model_name in MODELS_TO_TRY:
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt
                )
                print(f"✅ Used model: {model_name}")
                return response.text
            except Exception as e:
                err_str = str(e)
                if "503" in err_str or "UNAVAILABLE" in err_str:
                    print(f"⏳ {model_name} busy (attempt {attempt+1}), waiting 5s...")
                    time.sleep(5)
                elif "429" in err_str or "RESOURCE_EXHAUSTED" in err_str:
                    print(f"🚫 {model_name} out of quota, trying next model...")
                    break  # Skip to next model
                else:
                    print(f"❌ {model_name} unexpected error: {err_str[:100]}")
                    break  # Skip to next model
    raise Exception("All models failed!")

# Test
result = call_gemini("Say 'Hello, RAG project!' if you can hear me.")
print(f"\n🤖 Gemini says: {result}")

In [ ]:
import time
import re
import pandas as pd
from tqdm import tqdm

# Model fallback list (in case one gets busy)
MODELS_TO_TRY = [
    "gemini-2.5-flash-lite",
    "gemini-2.0-flash-lite",
    "gemini-2.0-flash-001",
    "gemini-flash-lite-latest",
]

def call_gemini(prompt, max_retries=2):
    """Call Gemini with auto-fallback. Returns text or None on total failure."""
    for model_name in MODELS_TO_TRY:
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt
                )
                return response.text, model_name
            except Exception as e:
                err = str(e)
                if "503" in err or "UNAVAILABLE" in err:
                    time.sleep(3)
                elif "429" in err or "RESOURCE_EXHAUSTED" in err:
                    break  # switch model
                else:
                    break
    return None, None

# The strict RAG prompt template
RAG_PROMPT = """You are a financial data assistant answering questions about U.S. Treasury Bulletins.

Use ONLY the context below to answer the question. Do not use outside knowledge.

CONTEXT:
{context}

QUESTION:
{question}

CRITICAL OUTPUT RULES:
1. Output ONLY the final answer — no explanation, no reasoning, no preamble.
2. Match the EXACT format the question requests (number, list, percentage, etc.).
3. If the answer is a number, include any unit text (e.g., "March 1977", "$23,918,635").
4. For multi-number answers, use the format from the question (e.g., "[1.2, 3.4]").
5. Keep the answer under 200 characters, on ONE line.
6. If the context is insufficient, output exactly: INSUFFICIENT_CONTEXT
7. Wrap your final answer in tags like this: <FINAL_ANSWER>your answer here</FINAL_ANSWER>

Now provide the answer:"""

def extract_final_answer(llm_text):
    """Pull the answer out of <FINAL_ANSWER>...</FINAL_ANSWER> tags."""
    if not llm_text:
        return ""
    match = re.search(r'<FINAL_ANSWER>(.*?)</FINAL_ANSWER>', llm_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    # Fallback: last non-empty line
    lines = [l.strip() for l in llm_text.split('\n') if l.strip()]
    return lines[-1] if lines else ""

# Simple scoring: does the GT value appear in the prediction?
def simple_score(gt, pred):
    """Loose scoring: True if all GT numbers appear in prediction."""
    if not pred or "INSUFFICIENT_CONTEXT" in pred:
        return False
    gt_str = str(gt).lower().replace(',', '').replace('$', '').replace('%', '')
    pred_str = str(pred).lower().replace(',', '').replace('$', '').replace('%', '')
    # Extract numbers from GT
    gt_nums = re.findall(r'-?\d+\.?\d*', gt_str)
    if not gt_nums:
        return gt_str.strip() in pred_str.strip()
    # All GT numbers must appear in prediction
    return all(num in pred_str for num in gt_nums)

# Load questions
df_q = pd.read_csv("./data/filtered_questions.csv")
print(f"📋 Running full RAG pipeline on {len(df_q)} questions...\n")

# Run full pipeline
results = []
for idx, row in tqdm(df_q.iterrows(), total=len(df_q)):
    question = row['question']
    gt_answer = str(row['answer'])

    # Step 1: Retrieve top-5 chunks
    q_emb = embed_model.encode([question]).tolist()
    retrieved = collection.query(query_embeddings=q_emb, n_results=5)

    # Build context from top chunks
    context = ""
    for doc, meta in zip(retrieved['documents'][0], retrieved['metadatas'][0]):
        context += f"[Source: {meta['source']}]\n{doc}\n\n"

    # Step 2: Ask Gemini
    prompt = RAG_PROMPT.format(context=context[:8000], question=question)
    llm_response, model_used = call_gemini(prompt)

    # Step 3: Extract and score
    final_answer = extract_final_answer(llm_response)
    is_correct = simple_score(gt_answer, final_answer)

    results.append({
        "uid": row['uid'],
        "difficulty": row['difficulty'],
        "question": question[:100] + "...",
        "ground_truth": gt_answer,
        "prediction": final_answer[:200],
        "correct": is_correct,
        "model": model_used,
    })

    time.sleep(1)  # Be nice to the free API

# Compute Factual Accuracy
df_results = pd.DataFrame(results)
correct = df_results['correct'].sum()
total = len(df_results)
factual_accuracy = correct / total

print(f"\n{'='*60}")
print(f"📊 BASELINE METRICS (Set B: Generator)")
print(f"{'='*60}")
print(f"Factual Accuracy: {factual_accuracy:.2%}  ({correct}/{total})")
print(f"{'='*60}\n")

print("📋 Per-question results:")
for r in results:
    status = "✅" if r['correct'] else "❌"
    print(f"\n{status} {r['uid']} ({r['difficulty']})")
    print(f"   GT:   {r['ground_truth']}")
    print(f"   Pred: {r['prediction'][:100]}")

# Save
df_results.to_csv("./data/baseline_full_results.csv", index=False)
print(f"\n💾 Saved to ./data/baseline_full_results.csv")

In [ ]:
import re
import os
import glob
import chromadb
from tqdm import tqdm

# =========================================================
# IMPROVED CHUNKING: Larger chunks + overlap + metadata
# =========================================================

CHUNK_SIZE = 1500   # 3x larger — keeps tables intact
OVERLAP = 300       # Overlap so split rows still appear in neighbor chunk

def extract_year_month(filename):
    """Pull year and month from filename like 'treasury_bulletin_2024_03.txt'."""
    match = re.search(r'(\d{4})_(\d{2})', filename)
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None

recent_years = ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
txt_folder = "./data/treasury_bulletins_parsed/transformed"

all_files = []
for year in recent_years:
    pattern = os.path.join(txt_folder, f"treasury_bulletin_{year}_*.txt")
    all_files.extend(glob.glob(pattern))
all_files = sorted(all_files)

# Build smarter chunks with metadata
eng_chunks = []
for filepath in all_files:
    filename = os.path.basename(filepath)
    year, month = extract_year_month(filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()

    # Larger chunks with overlap
    step = CHUNK_SIZE - OVERLAP
    for i in range(0, len(text), step):
        chunk_text = text[i:i+CHUNK_SIZE]
        if len(chunk_text.strip()) > 100:
            eng_chunks.append({
                "text": chunk_text,
                "source": filename,
                "year": year,
                "month": month,
                "chunk_id": f"{filename}_{i}"
            })

print(f"📦 ENGINEERED chunks: {len(eng_chunks)} (baseline had 17,727)")

# =========================================================
# Build NEW ChromaDB collection with year/month metadata
# =========================================================

client = chromadb.Client()
try:
    client.delete_collection("engineered_rag")
except:
    pass

eng_collection = client.create_collection(
    name="engineered_rag",
    metadata={"description": "Engineered: 1500-char chunks, 300 overlap, year+month metadata"}
)

BATCH_SIZE = 256
print(f"\n🔄 Embedding {len(eng_chunks)} engineered chunks...")
for i in tqdm(range(0, len(eng_chunks), BATCH_SIZE)):
    batch = eng_chunks[i:i+BATCH_SIZE]
    texts = [c["text"] for c in batch]
    embeddings = embed_model.encode(texts, show_progress_bar=False).tolist()
    eng_collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=[{"source": c["source"], "year": c["year"], "month": c["month"]} for c in batch],
        ids=[c["chunk_id"] for c in batch]
    )

print(f"\n✅ Engineered DB ready! Chunks: {eng_collection.count()}")

In [ ]:
import re
import pandas as pd

df_questions = pd.read_csv("./data/filtered_questions.csv")

def extract_year_month_from_query(query):
    """Detect year and month mentions in the question."""
    year_match = re.search(r'\b(20\d{2})\b', query)
    year = int(year_match.group(1)) if year_match else None

    months = {'january':1,'february':2,'march':3,'april':4,'may':5,'june':6,
              'july':7,'august':8,'september':9,'october':10,'november':11,'december':12}
    month = None
    q_lower = query.lower()
    for name, num in months.items():
        if name in q_lower:
            month = num
            break
    return year, month

def engineered_retrieve(query, k=5):
    """Retrieve with year/month metadata filter when available."""
    year, month = extract_year_month_from_query(query)

    where_filter = None
    if year and month:
        where_filter = {"$and": [{"year": year}, {"month": month}]}
    elif year:
        where_filter = {"year": year}

    q_emb = embed_model.encode([query]).tolist()
    try:
        results = eng_collection.query(
            query_embeddings=q_emb,
            n_results=k,
            where=where_filter
        )
        # Fallback: if filter returns nothing, retry without filter
        if not results['documents'][0]:
            results = eng_collection.query(query_embeddings=q_emb, n_results=k)
    except:
        results = eng_collection.query(query_embeddings=q_emb, n_results=k)
    return results

print("✅ Engineered retriever ready")

In [ ]:
import pandas as pd

df = pd.read_csv("./data/officeqa_full.csv")
recent_years = ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
pattern = '|'.join(recent_years)
df_touching = df[df['source_files'].str.contains(pattern, na=False)].copy()

def all_in_recent(s):
    if pd.isna(s):
        return False
    files = [f.strip() for f in str(s).split('\n') if f.strip()]
    return all(any(yr in f for yr in recent_years) for f in files)

df_final = df_touching[df_touching['source_files'].apply(all_in_recent)].copy()
df_final.to_csv("./data/filtered_questions.csv", index=False)
print(f"✅ Saved {len(df_final)} questions")

In [ ]:
# Install packages
!pip install -q huggingface_hub pandas chromadb sentence-transformers google-genai

# Import tools
from huggingface_hub import snapshot_download, login
from google.colab import userdata

# Login to Hugging Face
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# Download the .txt files
snapshot_download(
    repo_id="databricks/officeqa",
    repo_type="dataset",
    allow_patterns="treasury_bulletins_parsed/transformed/*.txt",
    local_dir="./data"
)

# Download the answer key CSV
snapshot_download(
    repo_id="databricks/officeqa",
    repo_type="dataset",
    allow_patterns="officeqa_full.csv",
    local_dir="./data"
)

print("✅ Download complete!")

In [ ]:
import os

print("📁 Files in ./data:")
for f in os.listdir("./data"):
    print("  -", f)

txt_folder = "./data/treasury_bulletins_parsed/transformed"
if os.path.exists(txt_folder):
    txt_files = os.listdir(txt_folder)
    print(f"\n📄 Found {len(txt_files)} .txt files")
else:
    print("❌ txt folder missing!")

In [ ]:
import pandas as pd

df = pd.read_csv("./data/officeqa_full.csv")
recent_years = ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
pattern = '|'.join(recent_years)
df_touching = df[df['source_files'].str.contains(pattern, na=False)].copy()

def all_in_recent(s):
    if pd.isna(s):
        return False
    files = [f.strip() for f in str(s).split('\n') if f.strip()]
    return all(any(yr in f for yr in recent_years) for f in files)

df_final = df_touching[df_touching['source_files'].apply(all_in_recent)].copy()
df_final.to_csv("./data/filtered_questions.csv", index=False)
print(f"✅ Saved {len(df_final)} questions to ./data/filtered_questions.csv")

In [ ]:
from sentence_transformers import SentenceTransformer

print("🤖 Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model ready!")

In [ ]:
import re
import os
import glob
import chromadb
from tqdm import tqdm

CHUNK_SIZE = 1500
OVERLAP = 300

def extract_year_month(filename):
    match = re.search(r'(\d{4})_(\d{2})', filename)
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None

recent_years = ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
txt_folder = "./data/treasury_bulletins_parsed/transformed"
all_files = []
for year in recent_years:
    pattern = os.path.join(txt_folder, f"treasury_bulletin_{year}_*.txt")
    all_files.extend(glob.glob(pattern))
all_files = sorted(all_files)

# Build chunks with metadata
eng_chunks = []
for filepath in all_files:
    filename = os.path.basename(filepath)
    year, month = extract_year_month(filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    step = CHUNK_SIZE - OVERLAP
    for i in range(0, len(text), step):
        chunk_text = text[i:i+CHUNK_SIZE]
        if len(chunk_text.strip()) > 100:
            eng_chunks.append({
                "text": chunk_text,
                "source": filename,
                "year": year,
                "month": month,
                "chunk_id": f"{filename}_{i}"
            })

print(f"📦 ENGINEERED chunks: {len(eng_chunks)}")

# Create ChromaDB collection
client = chromadb.Client()
try:
    client.delete_collection("engineered_rag")
except:
    pass

eng_collection = client.create_collection(
    name="engineered_rag",
    metadata={"description": "Engineered: 1500-char chunks, 300 overlap, year+month metadata"}
)

BATCH_SIZE = 256
print(f"\n🔄 Embedding {len(eng_chunks)} chunks...")
for i in tqdm(range(0, len(eng_chunks), BATCH_SIZE)):
    batch = eng_chunks[i:i+BATCH_SIZE]
    texts = [c["text"] for c in batch]
    embeddings = embed_model.encode(texts, show_progress_bar=False).tolist()
    eng_collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=[{"source": c["source"], "year": c["year"], "month": c["month"]} for c in batch],
        ids=[c["chunk_id"] for c in batch]
    )

print(f"\n✅ Engineered DB ready! Chunks: {eng_collection.count()}")

In [ ]:
import re
import pandas as pd

df_questions = pd.read_csv("./data/filtered_questions.csv")
print(f"✅ Loaded {len(df_questions)} questions")

def extract_year_month_from_query(query):
    year_match = re.search(r'\b(20\d{2})\b', query)
    year = int(year_match.group(1)) if year_match else None

    months = {'january':1,'february':2,'march':3,'april':4,'may':5,'june':6,
              'july':7,'august':8,'september':9,'october':10,'november':11,'december':12}
    month = None
    q_lower = query.lower()
    for name, num in months.items():
        if name in q_lower:
            month = num
            break
    return year, month

def engineered_retrieve(query, k=5):
    year, month = extract_year_month_from_query(query)
    where_filter = None
    if year and month:
        where_filter = {"$and": [{"year": year}, {"month": month}]}
    elif year:
        where_filter = {"year": year}

    q_emb = embed_model.encode([query]).tolist()
    try:
        results = eng_collection.query(
            query_embeddings=q_emb,
            n_results=k,
            where=where_filter
        )
        if not results['documents'][0]:
            results = eng_collection.query(query_embeddings=q_emb, n_results=k)
    except:
        results = eng_collection.query(query_embeddings=q_emb, n_results=k)
    return results

print("✅ Engineered retriever ready")

In [ ]:
def get_gt_files(source_str):
    if pd.isna(source_str):
        return set()
    return set(f.strip() for f in str(source_str).split('\n') if f.strip().endswith('.txt'))

hit_count = 0
mrr_sum = 0.0
eng_results_log = []
K = 5

print(f"📋 Testing {len(df_questions)} questions on ENGINEERED RAG...\n")

for idx, row in df_questions.iterrows():
    question = row['question']
    gt_files = get_gt_files(row['source_files'])

    results = engineered_retrieve(question, k=K)

    # Get unique sources in order
    retrieved_sources = []
    seen = set()
    for meta in results['metadatas'][0]:
        src = meta['source']
        if src not in seen:
            retrieved_sources.append(src)
            seen.add(src)

    hit = any(gt in retrieved_sources for gt in gt_files)
    if hit:
        hit_count += 1

    rank = 0
    for i, src in enumerate(retrieved_sources, start=1):
        if src in gt_files:
            rank = i
            break
    reciprocal_rank = 1.0/rank if rank > 0 else 0.0
    mrr_sum += reciprocal_rank

    eng_results_log.append({
        "uid": row['uid'],
        "difficulty": row['difficulty'],
        "gt_files": list(gt_files),
        "retrieved": retrieved_sources,
        "hit": hit,
        "rank_of_first_correct": rank,
        "reciprocal_rank": reciprocal_rank
    })

hit_rate = hit_count / len(df_questions)
mrr = mrr_sum / len(df_questions)

print("="*60)
print("📊 ENGINEERED METRICS (Set A: Retriever)")
print("="*60)
print(f"Hit Rate@{K}:  {hit_rate:.2%}  ({hit_count}/{len(df_questions)} questions)")
print(f"MRR:          {mrr:.4f}")
print("="*60)
print(f"\n📈 vs Baseline: Hit Rate 18.18% → {hit_rate:.2%},  MRR 0.1136 → {mrr:.4f}\n")

print("📋 Per-question breakdown:")
for r in eng_results_log:
    status = "✅" if r['hit'] else "❌"
    print(f"{status} {r['uid']} ({r['difficulty']}): rank={r['rank_of_first_correct']}, GT={r['gt_files']}")

pd.DataFrame(eng_results_log).to_csv("./data/engineered_retrieval_results.csv", index=False)
print("\n💾 Saved to ./data/engineered_retrieval_results.csv")

In [ ]:
!wget -q https://raw.githubusercontent.com/databricks/officeqa/main/reward.py -O reward.py
print("✅ Downloaded official reward.py")

from reward import score_answer
print("✅ score_answer function imported")

# Quick test
print("\n🧪 Quick tests:")
print("  Exact match ('2,602' vs '2,602'):", score_answer("2,602", "2,602", tolerance=0.01))
print("  Wrong answer:", score_answer("2,602", "5,000", tolerance=0.01))

In [ ]:
from google import genai
from google.colab import userdata
import time

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

MODELS_TO_TRY = [
    "gemini-2.5-flash-lite",
    "gemini-2.0-flash-lite",
    "gemini-2.0-flash-001",
    "gemini-flash-lite-latest",
]

def call_gemini(prompt, max_retries=2):
    for model_name in MODELS_TO_TRY:
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt
                )
                return response.text, model_name
            except Exception as e:
                err = str(e)
                if "503" in err or "UNAVAILABLE" in err:
                    time.sleep(3)
                elif "429" in err or "RESOURCE_EXHAUSTED" in err:
                    break
                else:
                    break
    return None, None

RAG_PROMPT = """You are a financial data assistant answering questions about U.S. Treasury Bulletins.

Use ONLY the context below to answer the question. Do not use outside knowledge.

CONTEXT:
{context}

QUESTION: {question}

CRITICAL OUTPUT RULES:
1. Output ONLY the final answer — no explanation, no reasoning, no preamble.
2. Match the EXACT format the question requests (number, list, percentage, etc.).
3. If the answer is a number, include any unit text (e.g., "March 1977", "$23,918,635").
4. For multi-number answers, use the format from the question (e.g., "[1.2, 3.4]").
5. Keep the answer under 200 characters, on ONE line.
6. If the context is insufficient, output exactly: INSUFFICIENT_CONTEXT
7. Wrap your final answer in tags like this: <FINAL_ANSWER>your answer here</FINAL_ANSWER>

Now provide the answer:"""

import re
def extract_final_answer(llm_text):
    if not llm_text:
        return ""
    match = re.search(r'<FINAL_ANSWER>(.*?)</FINAL_ANSWER>', llm_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    lines = [l.strip() for l in llm_text.split('\n') if l.strip()]
    return lines[-1] if lines else ""

print("✅ Gemini ready")

In [ ]:
from tqdm import tqdm

results = []
print(f"📋 Running full RAG pipeline on {len(df_questions)} questions (ENGINEERED)...\n")

for idx, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    question = row['question']
    gt_answer = str(row['answer'])

    # Step 1: Engineered retrieval (with metadata filter)
    retrieved = engineered_retrieve(question, k=5)

    # Step 2: Build context
    context = ""
    for doc, meta in zip(retrieved['documents'][0], retrieved['metadatas'][0]):
        context += f"[Source: {meta['source']}]\n{doc}\n\n"

    # Step 3: Ask Gemini
    prompt = RAG_PROMPT.format(context=context[:8000], question=question)
    llm_response, model_used = call_gemini(prompt)

    # Step 4: Extract & score with OFFICIAL scorer
    final_answer = extract_final_answer(llm_response)
    try:
        is_correct = score_answer(gt_answer, final_answer, tolerance=0.01)
    except Exception as e:
        is_correct = False

    results.append({
        "uid": row['uid'],
        "difficulty": row['difficulty'],
        "question": question[:100] + "...",
        "ground_truth": gt_answer,
        "prediction": final_answer[:200],
        "correct": is_correct,
        "model": model_used,
    })
    time.sleep(1)

df_eng_results = pd.DataFrame(results)
correct = df_eng_results['correct'].sum()
total = len(df_eng_results)
factual_accuracy = correct / total

print("\n" + "="*60)
print("📊 ENGINEERED METRICS (Set B: Generator)")
print("="*60)
print(f"Factual Accuracy: {factual_accuracy:.2%}  ({correct}/{total})")
print("="*60 + "\n")

for r in results:
    status = "✅" if r['correct'] else "❌"
    print(f"\n{status} {r['uid']} ({r['difficulty']})")
    print(f"   GT:   {r['ground_truth']}")
    print(f"   Pred: {r['prediction'][:100]}")

df_eng_results.to_csv("./data/engineered_full_results.csv", index=False)
print("\n💾 Saved to ./data/engineered_full_results.csv")

In [ ]:
from reward import score_answer

# Load baseline results (if the file still exists)
try:
    df_baseline = pd.read_csv("./data/baseline_full_results.csv")

    new_correct = 0
    for _, row in df_baseline.iterrows():
        try:
            if score_answer(str(row['ground_truth']), str(row['prediction']), tolerance=0.01):
                new_correct += 1
        except:
            pass

    new_accuracy = new_correct / len(df_baseline)
    print(f"📊 BASELINE re-scored with OFFICIAL reward.py:")
    print(f"   Old (my simple scorer):  0.00%")
    print(f"   New (official scorer):   {new_accuracy:.2%}  ({new_correct}/{len(df_baseline)})")
except FileNotFoundError:
    print("⚠️ baseline_full_results.csv not found — skip this cell")
    print("   (Colab session was reset, baseline results were wiped)")

In [ ]:
import re

def extract_numbers(text):
    """Pull out all numbers from a text string."""
    if not text:
        return []
    # Remove $, commas, %
    clean = str(text).replace(',', '').replace('$', '').replace('%', '')
    # Find all numbers (including decimals and negatives)
    numbers = re.findall(r'-?\d+\.?\d*', clean)
    return [n for n in numbers if n]  # remove empty

def check_groundedness(prediction, context):
    """Check if numbers in the prediction actually appear in the context."""
    if not prediction or "INSUFFICIENT_CONTEXT" in prediction:
        return None  # skip — no claim was made

    pred_numbers = extract_numbers(prediction)
    if not pred_numbers:
        return None  # no numeric claim to check

    # Clean context the same way
    context_clean = context.replace(',', '').replace('$', '').replace('%', '')

    # Check if each number appears in context
    supported = sum(1 for num in pred_numbers if num in context_clean)
    total = len(pred_numbers)

    return supported, total

# =====================================================
# Run groundedness analysis on ENGINEERED results
# =====================================================
print("🔍 Analyzing Groundedness & Hallucinations (ENGINEERED)...\n")

total_claims = 0
supported_claims = 0
answered_questions = 0

for idx, row in df_questions.iterrows():
    question = row['question']

    # Get the same context we gave to Gemini
    retrieved = engineered_retrieve(question, k=5)
    context = ""
    for doc, meta in zip(retrieved['documents'][0], retrieved['metadatas'][0]):
        context += f"[Source: {meta['source']}]\n{doc}\n\n"

    # Get our prediction from the earlier results
    pred_row = df_eng_results[df_eng_results['uid'] == row['uid']]
    if pred_row.empty:
        continue
    prediction = pred_row.iloc[0]['prediction']

    result = check_groundedness(prediction, context)
    if result is None:
        continue  # skip INSUFFICIENT_CONTEXT

    answered_questions += 1
    sup, tot = result
    supported_claims += sup
    total_claims += tot

if total_claims > 0:
    groundedness = supported_claims / total_claims
    hallucination_rate = 1 - groundedness
else:
    groundedness = 0
    hallucination_rate = 0

print("="*60)
print("📊 ENGINEERED — Groundedness & Hallucination")
print("="*60)
print(f"Questions with actual answers:  {answered_questions}/{len(df_questions)}")
print(f"Total numeric claims made:      {total_claims}")
print(f"Claims supported by context:    {supported_claims}")
print(f"Groundedness:                   {groundedness:.2%}")
print(f"Hallucination Rate:             {hallucination_rate:.2%}")
print("="*60)

In [ ]:
# Do the same for baseline
print("🔍 Analyzing Groundedness & Hallucinations (BASELINE)...\n")

# Try loading baseline results
try:
    df_baseline = pd.read_csv("./data/baseline_full_results.csv")

    # We need the baseline collection — but if it was wiped, all predictions were INSUFFICIENT_CONTEXT
    # So groundedness is undefined (no claims made)

    num_answered = sum(1 for _, r in df_baseline.iterrows()
                       if "INSUFFICIENT_CONTEXT" not in str(r['prediction']))

    if num_answered == 0:
        print("⚠️ Baseline had 0 answered questions (all INSUFFICIENT_CONTEXT)")
        print("   → Groundedness: N/A (no claims made)")
        print("   → Hallucination Rate: 0% (can't hallucinate if you don't answer)")
    else:
        print(f"Baseline answered {num_answered} questions")
        # Full analysis would need re-retrieval from baseline collection

except FileNotFoundError:
    print("⚠️ baseline_full_results.csv not found (Colab was reset)")

In [ ]:
# Pick a question where retrieval was SUCCESSFUL (rank=1)
# From your output: UID0066, UID0099, UID0102 all had rank 1 or 2

test_uid = "UID0066"  # An "easy" question with rank=1

# Get the question
row = df_questions[df_questions['uid'] == test_uid].iloc[0]
question = row['question']
gt_answer = row['answer']
gt_files = row['source_files']

print("="*70)
print(f"QUESTION ({test_uid}):")
print(question)
print(f"\nGROUND TRUTH ANSWER: {gt_answer}")
print(f"GROUND TRUTH SOURCE: {gt_files}")
print("="*70)

# Retrieve like we did before
retrieved = engineered_retrieve(question, k=5)

print("\n📄 WHAT GEMINI SAW (retrieved chunks):\n")
for i, (doc, meta) in enumerate(zip(retrieved['documents'][0], retrieved['metadatas'][0])):
    print(f"\n--- Chunk {i+1} from {meta['source']} ---")
    print(doc[:500])  # first 500 chars
    print("...")

# Now show Gemini's answer
pred_row = df_eng_results[df_eng_results['uid'] == test_uid].iloc[0]
print("\n" + "="*70)
print(f"GEMINI'S ANSWER: {pred_row['prediction']}")
print("="*70)

In [ ]:
# NEW, less strict prompt
RAG_PROMPT_V2 = """You are a financial data assistant answering questions about U.S. Treasury Bulletins.

Use the context below to find the answer. The answer IS in the context — look carefully at tables, numbers, and dates.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
1. Find the specific number, date, or value that answers the question.
2. Look at tables carefully — the answer may be in a row/column intersection.
3. Match the exact format the question requests (e.g., "$1,234", "March 2020", "1.234").
4. Give your best answer based on the context — do not say INSUFFICIENT_CONTEXT unless the topic is completely absent.
5. Output ONLY the final answer on one line. No explanation.
6. Wrap your final answer in tags: <FINAL_ANSWER>your answer here</FINAL_ANSWER>

Now provide the answer:"""

# Re-run the pipeline with the new prompt
from tqdm import tqdm
from reward import score_answer

results_v2 = []
print(f"📋 Re-running with IMPROVED prompt on {len(df_questions)} questions...\n")

for idx, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    question = row['question']
    gt_answer = str(row['answer'])

    retrieved = engineered_retrieve(question, k=5)

    context = ""
    for doc, meta in zip(retrieved['documents'][0], retrieved['metadatas'][0]):
        context += f"[Source: {meta['source']}]\n{doc}\n\n"

    prompt = RAG_PROMPT_V2.format(context=context[:8000], question=question)
    llm_response, model_used = call_gemini(prompt)

    final_answer = extract_final_answer(llm_response)
    try:
        is_correct = score_answer(gt_answer, final_answer, tolerance=0.01)
    except:
        is_correct = False

    results_v2.append({
        "uid": row['uid'],
        "difficulty": row['difficulty'],
        "question": question[:100] + "...",
        "ground_truth": gt_answer,
        "prediction": final_answer[:200],
        "correct": is_correct,
        "model": model_used,
    })
    time.sleep(1)

df_eng_results = pd.DataFrame(results_v2)  # Overwrite with new results
correct = df_eng_results['correct'].sum()
total = len(df_eng_results)
factual_accuracy = correct / total

print("\n" + "="*60)
print("📊 ENGINEERED METRICS v2 (Set B: Generator, Better Prompt)")
print("="*60)
print(f"Factual Accuracy: {factual_accuracy:.2%}  ({correct}/{total})")
print("="*60 + "\n")

for r in results_v2:
    status = "✅" if r['correct'] else "❌"
    print(f"\n{status} {r['uid']} ({r['difficulty']})")
    print(f"   GT:   {r['ground_truth']}")
    print(f"   Pred: {r['prediction'][:100]}")

df_eng_results.to_csv("./data/engineered_full_results.csv", index=False)
print("\n💾 Saved to ./data/engineered_full_results.csv")

In [ ]:
import re

def extract_numbers(text):
    if not text:
        return []
    clean = str(text).replace(',', '').replace('$', '').replace('%', '')
    numbers = re.findall(r'-?\d+\.?\d*', clean)
    return [n for n in numbers if n]

def check_groundedness(prediction, context):
    if not prediction or "INSUFFICIENT_CONTEXT" in prediction:
        return None
    pred_numbers = extract_numbers(prediction)
    if not pred_numbers:
        return None
    context_clean = context.replace(',', '').replace('$', '').replace('%', '')
    supported = sum(1 for num in pred_numbers if num in context_clean)
    total = len(pred_numbers)
    return supported, total

print("🔍 Analyzing Groundedness & Hallucinations (ENGINEERED v2)...\n")

total_claims = 0
supported_claims = 0
answered_questions = 0
per_question_log = []

for idx, row in df_questions.iterrows():
    question = row['question']

    retrieved = engineered_retrieve(question, k=5)
    context = ""
    for doc, meta in zip(retrieved['documents'][0], retrieved['metadatas'][0]):
        context += f"[Source: {meta['source']}]\n{doc}\n\n"

    pred_row = df_eng_results[df_eng_results['uid'] == row['uid']]
    if pred_row.empty:
        continue
    prediction = pred_row.iloc[0]['prediction']

    result = check_groundedness(prediction, context)
    if result is None:
        per_question_log.append((row['uid'], "SKIP (no claim)", 0, 0))
        continue

    answered_questions += 1
    sup, tot = result
    supported_claims += sup
    total_claims += tot
    per_question_log.append((row['uid'], prediction[:60], sup, tot))

groundedness = supported_claims / total_claims if total_claims > 0 else 0
hallucination_rate = 1 - groundedness if total_claims > 0 else 0

print("="*60)
print("📊 ENGINEERED — Groundedness & Hallucination")
print("="*60)
print(f"Questions with actual answers:  {answered_questions}/{len(df_questions)}")
print(f"Total numeric claims made:      {total_claims}")
print(f"Claims supported by context:    {supported_claims}")
print(f"Groundedness:                   {groundedness:.2%}")
print(f"Hallucination Rate:             {hallucination_rate:.2%}")
print("="*60)

print("\n📋 Per-question breakdown:")
for uid, pred, sup, tot in per_question_log:
    if tot == 0:
        print(f"  {uid}: {pred}")
    else:
        print(f"  {uid}: {sup}/{tot} claims grounded — Pred: {pred}")

In [ ]:
# Debug the 3 answered questions
for test_uid in ["UID0099", "UID0102", "UID0108"]:
    row = df_questions[df_questions['uid'] == test_uid].iloc[0]
    pred_row = df_eng_results[df_eng_results['uid'] == test_uid].iloc[0]
    prediction = pred_row['prediction']

    # Get context
    retrieved = engineered_retrieve(row['question'], k=5)
    context = ""
    for doc in retrieved['documents'][0]:
        context += doc + "\n\n"

    print(f"\n{'='*60}")
    print(f"{test_uid} — Prediction: {prediction}")
    print(f"{'='*60}")

    # Extract prediction number
    pred_nums = re.findall(r'-?\d+\.?\d*', prediction.replace(',', ''))
    print(f"Numbers in prediction: {pred_nums}")

    # Check if each number appears in the context (multiple formats)
    for num in pred_nums:
        context_clean = context.replace(',', '')
        num_stripped = num.rstrip('0').rstrip('.') if '.' in num else num  # 9162.00 → 9162

        found_exact = num in context_clean
        found_stripped = num_stripped in context_clean

        print(f"  '{num}' in context (exact)?    {found_exact}")
        print(f"  '{num_stripped}' in context (stripped)?  {found_stripped}")

In [ ]:
import re

def normalize_number(n):
    """Turn '9162.00' or '9,162' or '9162' all into '9162' for comparison."""
    s = str(n).replace(',', '').replace('$', '').replace('%', '').strip()
    try:
        f = float(s)
        # If it's a whole number, drop the decimal
        if f == int(f):
            return str(int(f))
        return str(f)
    except:
        return s

def extract_numbers(text):
    if not text:
        return []
    clean = str(text).replace(',', '').replace('$', '').replace('%', '')
    numbers = re.findall(r'-?\d+\.?\d*', clean)
    return [normalize_number(n) for n in numbers if n]

def context_contains_number(num, context):
    """Check if a number appears in context in ANY common format."""
    context_clean = context.replace(',', '').replace('$', '').replace('%', '')
    num_norm = normalize_number(num)

    # Try multiple formats
    variants = [
        num_norm,                          # "9162"
        f"{num_norm}.0",                   # "9162.0"
        f"{num_norm}.00",                  # "9162.00"
    ]
    # Also try as a float if possible
    try:
        f = float(num_norm)
        variants.append(str(f))
    except:
        pass

    return any(v in context_clean for v in variants)

def check_groundedness_v2(prediction, context):
    if not prediction or "INSUFFICIENT_CONTEXT" in prediction:
        return None
    pred_numbers = extract_numbers(prediction)
    if not pred_numbers:
        return None
    supported = sum(1 for num in pred_numbers if context_contains_number(num, context))
    return supported, len(pred_numbers)

# Re-run groundedness with the smarter checker
print("🔍 Analyzing Groundedness v2 (smarter matching)...\n")

total_claims = 0
supported_claims = 0
answered_questions = 0
per_question_log = []

for idx, row in df_questions.iterrows():
    question = row['question']
    retrieved = engineered_retrieve(question, k=5)
    context = ""
    for doc, meta in zip(retrieved['documents'][0], retrieved['metadatas'][0]):
        context += f"[Source: {meta['source']}]\n{doc}\n\n"

    pred_row = df_eng_results[df_eng_results['uid'] == row['uid']]
    if pred_row.empty:
        continue
    prediction = pred_row.iloc[0]['prediction']

    result = check_groundedness_v2(prediction, context)
    if result is None:
        per_question_log.append((row['uid'], "SKIP (no claim)", 0, 0))
        continue

    answered_questions += 1
    sup, tot = result
    supported_claims += sup
    total_claims += tot
    per_question_log.append((row['uid'], prediction[:60], sup, tot))

groundedness = supported_claims / total_claims if total_claims > 0 else 0
hallucination_rate = 1 - groundedness if total_claims > 0 else 0

print("="*60)
print("📊 ENGINEERED — Groundedness & Hallucination (v2)")
print("="*60)
print(f"Questions with actual answers:  {answered_questions}/{len(df_questions)}")
print(f"Total numeric claims made:      {total_claims}")
print(f"Claims supported by context:    {supported_claims}")
print(f"Groundedness:                   {groundedness:.2%}")
print(f"Hallucination Rate:             {hallucination_rate:.2%}")
print("="*60)

print("\n📋 Per-question breakdown:")
for uid, pred, sup, tot in per_question_log:
    if tot == 0:
        print(f"  {uid}: {pred}")
    else:
        status = "✅" if sup == tot else "❌"
        print(f"  {status} {uid}: {sup}/{tot} claims grounded — Pred: {pred}")

In [ ]:
import os
import glob
import chromadb
from tqdm import tqdm

# BASELINE settings: small chunks, no overlap, no metadata
recent_years = ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
txt_folder = "./data/treasury_bulletins_parsed/transformed"

all_files = []
for year in recent_years:
    pattern = os.path.join(txt_folder, f"treasury_bulletin_{year}_*.txt")
    all_files.extend(glob.glob(pattern))
all_files = sorted(all_files)

# BASELINE chunking: 500 chars, no overlap
baseline_chunks = []
for filepath in all_files:
    filename = os.path.basename(filepath)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    for i in range(0, len(text), 500):
        chunk_text = text[i:i+500]
        if len(chunk_text.strip()) > 50:
            baseline_chunks.append({
                "text": chunk_text,
                "source": filename,
                "chunk_id": f"{filename}_{i}"
            })

print(f"📦 BASELINE chunks: {len(baseline_chunks)}")

# Create baseline collection
client = chromadb.Client()
try:
    client.delete_collection("baseline_rag")
except:
    pass

baseline_collection = client.create_collection(
    name="baseline_rag",
    metadata={"description": "Baseline: 500-char chunks, no overlap, no metadata"}
)

BATCH_SIZE = 256
print(f"\n🔄 Embedding {len(baseline_chunks)} baseline chunks...")
for i in tqdm(range(0, len(baseline_chunks), BATCH_SIZE)):
    batch = baseline_chunks[i:i+BATCH_SIZE]
    texts = [c["text"] for c in batch]
    embeddings = embed_model.encode(texts, show_progress_bar=False).tolist()
    baseline_collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=[{"source": c["source"]} for c in batch],
        ids=[c["chunk_id"] for c in batch]
    )

print(f"\n✅ Baseline DB ready! Chunks: {baseline_collection.count()}")

In [ ]:
def get_gt_files(source_str):
    if pd.isna(source_str):
        return set()
    return set(f.strip() for f in str(source_str).split('\n') if f.strip().endswith('.txt'))

hit_count = 0
mrr_sum = 0.0
baseline_retrieval_log = []
K = 5

print(f"📋 Testing {len(df_questions)} questions on BASELINE RAG...\n")

for idx, row in df_questions.iterrows():
    question = row['question']
    gt_files = get_gt_files(row['source_files'])

    # SIMPLE retrieval — no metadata filtering
    q_emb = embed_model.encode([question]).tolist()
    results = baseline_collection.query(query_embeddings=q_emb, n_results=K)

    retrieved_sources = []
    seen = set()
    for meta in results['metadatas'][0]:
        src = meta['source']
        if src not in seen:
            retrieved_sources.append(src)
            seen.add(src)

    hit = any(gt in retrieved_sources for gt in gt_files)
    if hit:
        hit_count += 1

    rank = 0
    for i, src in enumerate(retrieved_sources, start=1):
        if src in gt_files:
            rank = i
            break
    reciprocal_rank = 1.0/rank if rank > 0 else 0.0
    mrr_sum += reciprocal_rank

    baseline_retrieval_log.append({
        "uid": row['uid'],
        "hit": hit,
        "rank": rank,
    })

baseline_hit_rate = hit_count / len(df_questions)
baseline_mrr = mrr_sum / len(df_questions)

print("="*60)
print("📊 BASELINE METRICS (Set A: Retriever)")
print("="*60)
print(f"Hit Rate@{K}:  {baseline_hit_rate:.2%}  ({hit_count}/{len(df_questions)})")
print(f"MRR:          {baseline_mrr:.4f}")
print("="*60)

In [ ]:
import time
from tqdm import tqdm
from reward import score_answer

# Same improved prompt used for engineered
RAG_PROMPT_V2 = """You are a financial data assistant answering questions about U.S. Treasury Bulletins.

Use the context below to find the answer. The answer IS in the context — look carefully at tables, numbers, and dates.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
1. Find the specific number, date, or value that answers the question.
2. Look at tables carefully — the answer may be in a row/column intersection.
3. Match the exact format the question requests (e.g., "$1,234", "March 2020", "1.234").
4. Give your best answer based on the context — do not say INSUFFICIENT_CONTEXT unless the topic is completely absent.
5. Output ONLY the final answer on one line. No explanation.
6. Wrap your final answer in tags: <FINAL_ANSWER>your answer here</FINAL_ANSWER>

Now provide the answer:"""

baseline_results = []
print(f"📋 Running BASELINE RAG with improved prompt on {len(df_questions)} questions...\n")

for idx, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    question = row['question']
    gt_answer = str(row['answer'])

    # BASELINE retrieval — no metadata filter
    q_emb = embed_model.encode([question]).tolist()
    retrieved = baseline_collection.query(query_embeddings=q_emb, n_results=5)

    context = ""
    for doc, meta in zip(retrieved['documents'][0], retrieved['metadatas'][0]):
        context += f"[Source: {meta['source']}]\n{doc}\n\n"

    prompt = RAG_PROMPT_V2.format(context=context[:8000], question=question)
    llm_response, model_used = call_gemini(prompt)

    final_answer = extract_final_answer(llm_response)
    try:
        is_correct = score_answer(gt_answer, final_answer, tolerance=0.01)
    except:
        is_correct = False

    baseline_results.append({
        "uid": row['uid'],
        "difficulty": row['difficulty'],
        "question": question[:100] + "...",
        "ground_truth": gt_answer,
        "prediction": final_answer[:200],
        "correct": is_correct,
        "model": model_used,
    })
    time.sleep(1)

df_baseline_results = pd.DataFrame(baseline_results)
correct = df_baseline_results['correct'].sum()
total = len(df_baseline_results)
baseline_accuracy = correct / total

print("\n" + "="*60)
print("📊 BASELINE METRICS (Set B: Generator, Improved Prompt)")
print("="*60)
print(f"Factual Accuracy: {baseline_accuracy:.2%}  ({correct}/{total})")
print("="*60 + "\n")

for r in baseline_results:
    status = "✅" if r['correct'] else "❌"
    print(f"\n{status} {r['uid']} ({r['difficulty']})")
    print(f"   GT:   {r['ground_truth']}")
    print(f"   Pred: {r['prediction'][:100]}")

df_baseline_results.to_csv("./data/baseline_full_results.csv", index=False)
print("\n💾 Saved to ./data/baseline_full_results.csv")

In [ ]:
import re

def normalize_number(n):
    s = str(n).replace(',', '').replace('$', '').replace('%', '').strip()
    try:
        f = float(s)
        if f == int(f):
            return str(int(f))
        return str(f)
    except:
        return s

def extract_numbers(text):
    if not text:
        return []
    clean = str(text).replace(',', '').replace('$', '').replace('%', '')
    numbers = re.findall(r'-?\d+\.?\d*', clean)
    return [normalize_number(n) for n in numbers if n]

def context_contains_number(num, context):
    context_clean = context.replace(',', '').replace('$', '').replace('%', '')
    num_norm = normalize_number(num)
    variants = [num_norm, f"{num_norm}.0", f"{num_norm}.00"]
    try:
        f = float(num_norm)
        variants.append(str(f))
    except:
        pass
    return any(v in context_clean for v in variants)

def check_groundedness_v2(prediction, context):
    if not prediction or "INSUFFICIENT_CONTEXT" in prediction:
        return None
    pred_numbers = extract_numbers(prediction)
    if not pred_numbers:
        return None
    supported = sum(1 for num in pred_numbers if context_contains_number(num, context))
    return supported, len(pred_numbers)

print("🔍 Analyzing Groundedness & Hallucination (BASELINE)...\n")

total_claims = 0
supported_claims = 0
answered_questions = 0
per_question_log = []

for idx, row in df_questions.iterrows():
    question = row['question']

    # BASELINE retrieval — same as generator used
    q_emb = embed_model.encode([question]).tolist()
    retrieved = baseline_collection.query(query_embeddings=q_emb, n_results=5)
    context = ""
    for doc, meta in zip(retrieved['documents'][0], retrieved['metadatas'][0]):
        context += f"[Source: {meta['source']}]\n{doc}\n\n"

    pred_row = df_baseline_results[df_baseline_results['uid'] == row['uid']]
    if pred_row.empty:
        continue
    prediction = pred_row.iloc[0]['prediction']

    result = check_groundedness_v2(prediction, context)
    if result is None:
        per_question_log.append((row['uid'], "SKIP (no claim)", 0, 0))
        continue

    answered_questions += 1
    sup, tot = result
    sup

In [ ]:
import re

def normalize_number(n):
    s = str(n).replace(',', '').replace('$', '').replace('%', '').strip()
    try:
        f = float(s)
        if f == int(f):
            return str(int(f))
        return str(f)
    except:
        return s

def extract_numbers(text):
    if not text:
        return []
    clean = str(text).replace(',', '').replace('$', '').replace('%', '')
    numbers = re.findall(r'-?\d+\.?\d*', clean)
    return [normalize_number(n) for n in numbers if n]

def context_contains_number(num, context):
    context_clean = context.replace(',', '').replace('$', '').replace('%', '')
    num_norm = normalize_number(num)
    variants = [num_norm, f"{num_norm}.0", f"{num_norm}.00"]
    try:
        f = float(num_norm)
        variants.append(str(f))
    except:
        pass
    return any(v in context_clean for v in variants)

def check_groundedness_v2(prediction, context):
    if not prediction or "INSUFFICIENT_CONTEXT" in prediction:
        return None
    pred_numbers = extract_numbers(prediction)
    if not pred_numbers:
        return None
    supported = sum(1 for num in pred_numbers if context_contains_number(num, context))
    return supported, len(pred_numbers)

print("🔍 Analyzing Groundedness & Hallucination (BASELINE)...\n")

total_claims = 0
supported_claims = 0
answered_questions = 0
per_question_log = []

for idx, row in df_questions.iterrows():
    question = row['question']

    # BASELINE retrieval — same as generator used
    q_emb = embed_model.encode([question]).tolist()
    retrieved = baseline_collection.query(query_embeddings=q_emb, n_results=5)
    context = ""
    for doc, meta in zip(retrieved['documents'][0], retrieved['metadatas'][0]):
        context += f"[Source: {meta['source']}]\n{doc}\n\n"

    pred_row = df_baseline_results[df_baseline_results['uid'] == row['uid']]
    if pred_row.empty:
        continue
    prediction = pred_row.iloc[0]['prediction']

    result = check_groundedness_v2(prediction, context)
    if result is None:
        per_question_log.append((row['uid'], "SKIP (no claim)", 0, 0))
        continue

    answered_questions += 1
    sup, tot = result
    supported_claims += sup
    total_claims += tot
    per_question_log.append((row['uid'], prediction[:60], sup, tot))

# Compute final metrics
baseline_groundedness = supported_claims / total_claims if total_claims > 0 else 0
baseline_hallucination_rate = 1 - baseline_groundedness if total_claims > 0 else 0

print("="*60)
print("📊 BASELINE — Groundedness & Hallucination")
print("="*60)
print(f"Questions with actual answers:  {answered_questions}/{len(df_questions)}")
print(f"Total numeric claims made:      {total_claims}")
print(f"Claims supported by context:    {supported_claims}")
print(f"Groundedness:                   {baseline_groundedness:.2%}")
print(f"Hallucination Rate:             {baseline_hallucination_rate:.2%}")
print("="*60)

print("\n📋 Per-question breakdown:")
for uid, pred, sup, tot in per_question_log:
    if tot == 0:
        print(f"  {uid}: {pred}")
    else:
        status = "✅" if sup == tot else "❌"
        print(f"  {status} {uid}: {sup}/{tot} claims grounded — Pred: {pred}")

In [ ]:
# Check baseline predictions
print("📋 Baseline predictions summary:\n")
print(df_baseline_results[['uid', 'prediction']].to_string(index=False))

print("\n\nHow many said INSUFFICIENT_CONTEXT?")
count_insuf = df_baseline_results['prediction'].str.contains("INSUFFICIENT_CONTEXT", na=False).sum()
print(f"  {count_insuf} / {len(df_baseline_results)}")

In [ ]:
import time
from tqdm import tqdm
from reward import score_answer

RAG_PROMPT_V2 = """You are a financial data assistant answering questions about U.S. Treasury Bulletins.

Use the context below to find the answer. The answer IS in the context — look carefully at tables, numbers, and dates.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
1. Find the specific number, date, or value that answers the question.
2. Look at tables carefully — the answer may be in a row/column intersection.
3. Match the exact format the question requests (e.g., "$1,234", "March 2020", "1.234").
4. Give your best answer based on the context — do not say INSUFFICIENT_CONTEXT unless the topic is completely absent.
5. Output ONLY the final answer on one line. No explanation.
6. Wrap your final answer in tags: <FINAL_ANSWER>your answer here</FINAL_ANSWER>

Now provide the answer:"""

baseline_results = []
print(f"📋 Running BASELINE RAG with improved prompt on {len(df_questions)} questions...\n")

for idx, row in df_questions.iterrows():
    question = row['question']
    gt_answer = str(row['answer'])

    # BASELINE retrieval — no metadata filter
    q_emb = embed_model.encode([question]).tolist()
    retrieved = baseline_collection.query(query_embeddings=q_emb, n_results=5)

    context = ""
    for doc, meta in zip(retrieved['documents'][0], retrieved['metadatas'][0]):
        context += f"[Source: {meta['source']}]\n{doc}\n\n"

    prompt = RAG_PROMPT_V2.format(context=context[:8000], question=question)
    llm_response, model_used = call_gemini(prompt)

    # 🆕 Debug: print what we got
    print(f"\n{row['uid']}:")
    print(f"  Model used: {model_used}")
    print(f"  Raw response: {str(llm_response)[:150] if llm_response else 'NONE (API failed!)'}")

    final_answer = extract_final_answer(llm_response) if llm_response else ""
    print(f"  Extracted:   {final_answer[:100]}")

    try:
        is_correct = score_answer(gt_answer, final_answer, tolerance=0.01)
    except:
        is_correct = False

    baseline_results.append({
        "uid": row['uid'],
        "difficulty": row['difficulty'],
        "question": question[:100] + "...",
        "ground_truth": gt_answer,
        "prediction": final_answer[:200] if final_answer else "API_ERROR",
        "correct": is_correct,
        "model": model_used,
    })
    time.sleep(2)  # 🆕 slower to avoid rate limits

df_baseline_results = pd.DataFrame(baseline_results)
correct = df_baseline_results['correct'].sum()
total = len(df_baseline_results)
baseline_accuracy = correct / total

print("\n" + "="*60)
print("📊 BASELINE METRICS (Set B: Generator)")
print("="*60)
print(f"Factual Accuracy: {baseline_accuracy:.2%}  ({correct}/{total})")
print("="*60)

df_baseline_results.to_csv("./data/baseline_full_results.csv", index=False)
print("\n💾 Saved")

In [ ]:
result = call_gemini("Say hello")
print("Test result:", result)